# Pakistan Property Price Analysis
## Zameen.com Dataset | EDA → Cleaning → Baseline Model

In [ ]:
# This cell installs all required Python packages.
# It works whether you're running locally, on Colab, or on Kaggle.
# If a package is already installed, pip skips it — so it's safe to re-run.
import subprocess, sys

# sys.executable points to the exact Python binary running this notebook.
# Using it ensures pip installs into the same environment, not a different one.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'pandas',       # data manipulation and analysis
    'numpy',        # fast numerical operations
    'matplotlib',   # base plotting library
    'seaborn',      # higher-level charts built on matplotlib
    'scikit-learn', # train/test split and error metrics
    'lightgbm',     # our prediction model
    'xgboost',      # gradient boosting (LightGBM's closest rival)
    'catboost',     # Yandex gradient boosting, strong with categoricals
    'shap',         # model interpretability — explain individual predictions
    '-q'            # quiet mode: suppresses verbose install output
])
print('All packages ready.')

In [ ]:
# ── Core data libraries ──────────────────────────────────────────────────
import pandas as pd                        # DataFrames: reading, filtering, grouping data
import numpy as np                         # Numerical operations: log, exp, arrays
import matplotlib.pyplot as plt            # Drawing charts and figures
import matplotlib.ticker as mticker        # Custom axis labels (e.g. show '50M' instead of '50000000')
import seaborn as sns                      # Prettier statistical charts (heatmaps, boxplots)
import re                                  # Regular expressions — used to parse '5 Marla' strings
import warnings                            # Used to suppress non-critical warning messages

# ── Machine learning libraries ───────────────────────────────────────────
from sklearn.metrics import mean_squared_error  # Computes RMSE (prediction error)
from sklearn.model_selection import train_test_split  # Splits data into train/validation sets
from sklearn.cluster import KMeans         # Geographic zone clustering on lat/lon
import lightgbm as lgb                    # LightGBM: our gradient boosting model for price prediction

# ── Display and style settings ───────────────────────────────────────────
warnings.filterwarnings('ignore')          # Hide non-critical warnings to keep output clean
pd.set_option('display.max_columns', 30)   # Show up to 30 columns when printing a DataFrame
pd.set_option('display.float_format', '{:,.2f}'.format)  # Format numbers with commas and 2 decimals
sns.set_theme(style='whitegrid', palette='deep')  # Clean white background with grid lines
plt.rcParams['figure.dpi'] = 100           # Higher resolution charts
plt.rcParams['figure.figsize'] = (10, 5)  # Default chart size: 10 inches wide, 5 inches tall

# ── Constants used throughout the notebook ───────────────────────────────
CSV_PATH = 'Pakistan House Prices and Property Listings.csv'  # Path to our raw data file
RANDOM_STATE = 42  # Fixed seed so results are reproducible — any number works, 42 is convention
# ── Pakistani consumer helpers ──────────────────────────────────────────
import shap
from math import radians, cos, sin, asin, sqrt

def format_pkr(amount):
    """Convert raw PKR to Lakh/Crore — how Pakistanis naturally express money."""
    crore = amount / 1e7
    lakh  = amount / 1e5
    return f'{crore:.2f} Crore' if crore >= 1 else f'{lakh:.1f} Lakh'

def haversine_km(lat1, lon1, lat2, lon2):
    """Straight-line distance in km between two GPS coordinates."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return 2 * R * asin(sqrt(a))

CITY_CENTERS = {           # commercial/business centre of each city
    'Karachi':    (24.8607, 67.0011),  # Clifton
    'Lahore':     (31.5204, 74.3587),  # The Mall
    'Islamabad':  (33.7215, 73.0433),  # Blue Area
    'Rawalpindi': (33.5651, 73.0169),  # Saddar
    'Faisalabad': (31.4504, 73.1350),  # Clock Tower
}

PREMIUM_KEYWORDS = [
    'dha', 'defence', 'bahria', 'gulberg', 'clifton', 'model town',
    'garden town', 'cavalry', 'pechs', 'emaar', 'park view',
    'f-6', 'f-7', 'f-8', 'f-10', 'f-11',   # Islamabad F-series sectors
    'e-7', 'e-8', 'e-11',                    # Islamabad E-series sectors
]

---
# Part 1: Exploratory Data Analysis
---

In [ ]:
# Load the entire CSV into a DataFrame called df_raw.
# We keep this as the 'untouched original' — all cleaning happens on a copy later.
df_raw = pd.read_csv(CSV_PATH)

# Print how many rows and columns the dataset has.
# :, formats the number with commas e.g. 168,446
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

# Show how much RAM the dataset occupies.
# deep=True counts the actual string memory, not just pointers.
print(f'Memory: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

# Print the data type of every column (int, float, object/text).
# 'object' dtype means it's a text/string column.
print(f'\nDtypes:\n{df_raw.dtypes}')

In [ ]:
# Count proper NaN (Not a Number) values — pandas' standard for missing data.
null_counts = df_raw.isnull().sum()

# Also count empty strings ('') — some tools store missing data as blank text
# instead of NaN, and pandas won't catch those with isnull() alone.
empty_counts = (df_raw == '').sum()

# Only print columns that actually have missing values (filter out the zeros).
print('Null counts (proper NaN):')
print(null_counts[null_counts > 0])

print('\nEmpty string counts (stored as blank text):')
print(empty_counts[empty_counts > 0])

In [ ]:
# Create a figure with 1 row and 2 side-by-side charts.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left chart: raw price histogram ──────────────────────────────────────
# .clip(upper=...) caps extreme values at the 99th percentile so a few
# billion-rupee outliers don't squash the rest of the chart into a thin sliver.
axes[0].hist(
    df_raw['price'].clip(upper=df_raw['price'].quantile(0.99)),
    bins=60,              # divide the range into 60 equal-width bars
    color='steelblue',
    edgecolor='white'     # white border between bars for readability
)
axes[0].set_title('Raw Price — All 168,446 Listings (capped at P99)')
axes[0].set_xlabel('Price (PKR)')

# Format the x-axis to show '50M' instead of '50000000'.
# The lambda takes a number x and returns a string like '50M'.
axes[0].xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M')
)
axes[0].set_ylabel('Count')

# ── Right chart: log-transformed price histogram ──────────────────────────
# np.log1p(x) = log(1 + x). The +1 safely handles zero values.
# This squishes the long right tail and makes the distribution bell-shaped,
# which is what machine learning models work best with.
axes[1].hist(np.log1p(df_raw['price']), bins=60, color='coral', edgecolor='white')
axes[1].set_title('log1p(Price) — All Listings')
axes[1].set_xlabel('log1p(Price)')
axes[1].set_ylabel('Count')

plt.suptitle('Price Distribution — All 168,446 Rows (Sale + Rent combined)', y=1.02)
plt.tight_layout()  # prevents chart titles from overlapping
plt.show()

# Print the skewness score.
# Raw > 1 confirms the distribution is heavily right-skewed.
# Log near 0 confirms the log transform fixed it.
print(f'Raw skewness:  {df_raw["price"].skew():.2f}  (> 1 means right-skewed)')
print(f'Log skewness:  {np.log1p(df_raw["price"]).skew():.2f}  (near 0 = more symmetric)')

In [ ]:
# ─── FOR SALE: Price distribution (raw and log-transformed) ─────────────
# Left chart: raw PKR prices. Capped at P99 so extreme outliers
# don't squish the x-axis and make the chart unreadable.
# Right chart: log1p(price). This converts the right-skewed distribution
# into a near-Gaussian (bell curve), which is what our model will actually predict.

sale = df_raw[df_raw['purpose'] == 'For Sale']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw price capped at 99th percentile
axes[0].hist(
    sale['price'].clip(upper=sale['price'].quantile(0.99)),
    bins=60, color='steelblue', edgecolor='white'
)
axes[0].set_title('For Sale — Raw Price (capped at P99)')
axes[0].set_xlabel('Price (PKR)')
# Format x-axis in millions so it reads '5M' instead of '5000000'
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

# Right: log-transformed price
axes[1].hist(np.log1p(sale['price']), bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('For Sale — log1p(Price)')
axes[1].set_xlabel('log1p(Price)')

plt.suptitle(f'For Sale Price Distribution  ({len(sale):,} listings)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('For Sale price statistics:')
print(sale['price'].describe().apply(lambda x: f'{x:,.0f}'))
print(f'\nSkewness raw:  {sale["price"].skew():.2f}')
print(f'Skewness log:  {np.log1p(sale["price"]).skew():.2f}')


In [ ]:
# ─── FOR RENT: Price distribution (raw and log-transformed) ─────────────
# Rental prices are in PKR per month — a completely different scale from sale prices.
# A 5-Marla house might sell for 15M PKR but rent for 30,000 PKR/month.
# This is why 'purpose' must be an input feature: the model needs to know
# whether it is predicting a sale price or a rental price.

rent = df_raw[df_raw['purpose'] == 'For Rent']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw price capped at 99th percentile
axes[0].hist(
    rent['price'].clip(upper=rent['price'].quantile(0.99)),
    bins=60, color='coral', edgecolor='white'
)
axes[0].set_title('For Rent — Raw Price/month (capped at P99)')
axes[0].set_xlabel('Price (PKR/month)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Right: log-transformed price
axes[1].hist(np.log1p(rent['price']), bins=60, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('For Rent — log1p(Price)')
axes[1].set_xlabel('log1p(Price)')

plt.suptitle(f'For Rent Price Distribution  ({len(rent):,} listings)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('For Rent price statistics (PKR/month):')
print(rent['price'].describe().apply(lambda x: f'{x:,.0f}'))
print(f'\nSkewness raw:  {rent["price"].skew():.2f}')
print(f'Skewness log:  {np.log1p(rent["price"]).skew():.2f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: city bar chart ─────────────────────────────────────────────────
# value_counts() counts how many listings belong to each city,
# and sorts from most to least.
city_counts = df_raw['city'].value_counts()
city_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Listings by City (all purposes)')
axes[0].set_xlabel('City')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)  # keep city names horizontal

# Annotate each bar with its exact count at the top.
for i, v in enumerate(city_counts):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

# ── Right: property type pie chart ───────────────────────────────────────
pt_counts = df_raw['property_type'].value_counts()
axes[1].pie(
    pt_counts,
    labels=pt_counts.index,
    autopct='%1.1f%%',   # show percentage with 1 decimal place on each slice
    startangle=140       # rotate the chart so the largest slice is on top-left
)
axes[1].set_title('Property Type Share (all purposes)')

plt.tight_layout()
plt.show()

# Print cardinality (number of unique values) for key categorical columns.
# Low cardinality = easy to encode for the model.
print(f'Unique cities: {df_raw["city"].nunique()}')
print(f'Unique property types: {df_raw["property_type"].nunique()}')
print(f'Unique provinces: {df_raw["province_name"].nunique()}')

In [ ]:
# The 'area' column stores values as strings like '5 Marla' or '2 Kanal'.
# This function extracts just the unit part (Marla or Kanal).
def extract_unit(s):
    # re.search scans the string for 'Marla' or 'Kanal' (case-insensitive).
    m = re.search(r'(Marla|Kanal)', str(s), re.IGNORECASE)
    # If found, return it with consistent capitalisation. Otherwise return 'Unknown'.
    return m.group(1).title() if m else 'Unknown'

# Apply the function to every row in the area column.
area_units = df_raw['area'].apply(extract_unit)

print('Area unit distribution:')
print(area_units.value_counts())

# Print the conversion rates we'll use in the cleaning step.
print(f'\nConversions used:')
print('  1 Marla = 272.25 sqft')
print('  1 Kanal = 5,445 sqft  (= 20 Marla)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: bedroom distribution ───────────────────────────────────────────
# value_counts() counts each unique value, sort_index() orders by bedroom number.
bed_counts = df_raw['bedrooms'].value_counts().sort_index()

# Only show listings with up to 15 bedrooms to keep the chart readable.
# Properties with 50+ bedrooms are likely data entry errors.
bed_counts[bed_counts.index <= 15].plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Bedroom Distribution (all listings, capped at 15)')
axes[0].set_xlabel('Bedrooms')
axes[0].set_ylabel('Count')

# ── Right: bath distribution ─────────────────────────────────────────────
bath_counts = df_raw['baths'].value_counts().sort_index()
bath_counts[bath_counts.index <= 15].plot(
    kind='bar', ax=axes[1], color='coral', edgecolor='white'
)
axes[1].set_title('Bath Distribution (all listings, capped at 15)')
axes[1].set_xlabel('Baths')
axes[1].set_ylabel('Count')

plt.suptitle(
    'Bedrooms & Baths — the large 0-value bars are UNFILLED fields, not true zeros',
    y=1.02
)
plt.tight_layout()
plt.show()

# Quantify how many listings have 0 values — these will become NaN in cleaning.
print(f'Bedrooms == 0: {(df_raw["bedrooms"] == 0).sum():,}  ({(df_raw["bedrooms"] == 0).mean()*100:.1f}% of all rows)')
print(f'Baths == 0:    {(df_raw["baths"] == 0).sum():,}  ({(df_raw["baths"] == 0).mean()*100:.1f}% of all rows)')
print('\nThese will be converted to NaN (missing) in the cleaning step.')
print('We do NOT drop these rows — LightGBM handles missing values natively.')

In [ ]:
# Filter to listings within Pakistan's geographic bounding box.
# Pakistan sits roughly between latitude 20–40°N and longitude 55–80°E.
# 5 rows have clearly wrong coordinates (e.g. lat=0 or swapped values) — we drop those.
geo = df_raw[
    df_raw['latitude'].between(20, 40) &
    df_raw['longitude'].between(55, 80) &
    (df_raw['price'] > 0)  # exclude the 2 zero-price rows
].copy()

# Compute log price for colour-coding each dot on the map.
geo['log_price'] = np.log1p(geo['price'])

fig, ax = plt.subplots(figsize=(12, 10))

# Draw a scatter plot where each dot = one listing.
# longitude goes on the x-axis (east-west), latitude on y-axis (north-south).
scatter = ax.scatter(
    geo['longitude'], geo['latitude'],
    c=geo['log_price'],   # colour each dot by its log-price
    cmap='RdYlGn',        # colour scale: green=cheap, yellow=mid, red=expensive
    alpha=0.25,           # 25% opacity so overlapping dots don't form a solid blob
    s=1.5,                # tiny dot size because we have 160k+ points
    vmin=geo['log_price'].quantile(0.02),  # colour scale starts at 2nd percentile
    vmax=geo['log_price'].quantile(0.98)   # colour scale ends at 98th percentile
)

# Add a colour bar legend on the right side of the chart.
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('log1p(Price) — green=cheap, red=expensive', fontsize=11)

# Annotate the approximate locations of major cities.
cities = {
    'Karachi':    (67.01, 24.86),
    'Lahore':     (74.34, 31.52),
    'Islamabad':  (73.05, 33.72),
    'Rawalpindi': (73.04, 33.40),
    'Faisalabad': (73.09, 31.42),
}
for name, (lon, lat) in cities.items():
    ax.annotate(
        name, (lon, lat),
        fontsize=11, fontweight='bold', color='navy',
        xytext=(6, 6), textcoords='offset points'  # nudge the label slightly off the point
    )

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title(
    'Geographic Price Heatmap — each dot is a listing, colour = log-price\n'
    'Lat/lon replace complex area-name text encoding entirely',
    fontsize=13
)
plt.tight_layout()
plt.show()

print(f'Points plotted: {len(geo):,}')
print(f'Rows excluded (bad coordinates): {len(df_raw) - len(geo)}')

In [ ]:
# Make a copy so we don't modify df_raw.
df_dated = df_raw.copy()

# Convert the date_added column from a string ('10/20/2018') to an actual date object.
df_dated['date_parsed'] = pd.to_datetime(df_dated['date_added'], format='%m/%d/%Y')

# Convert each date to its year-month period (e.g. 2019-06) for grouping.
df_dated['year_month'] = df_dated['date_parsed'].dt.to_period('M')

# Count how many listings were added in each month.
monthly = df_dated.groupby('year_month').size()

fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')

# Draw a red dashed vertical line at the train/test boundary.
# Everything to the left of this line = training data.
# Everything to the right (July 2019) = test data the model never saw during training.
ax.axvline(
    x=len(monthly) - 1.5,  # position just before the last bar (July 2019)
    color='red', linestyle='--', linewidth=2,
    label='Train / Test split (Jul 2019 = Test)'
)
ax.set_title('Listings Added per Month — All Purposes')
ax.set_xlabel('Month')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print('Monthly breakdown:')
print(monthly.to_string())

In [ ]:
# Add log_price to a copy of the raw data for correlation analysis.
df_corr = df_raw.copy()
df_corr['log_price'] = np.log1p(df_corr['price'])

# Select only the numeric columns we want to correlate.
# We use log_price (not raw price) so the scale doesn't distort correlations.
numeric_cols = ['log_price', 'baths', 'bedrooms', 'latitude', 'longitude']
corr = df_corr[numeric_cols].corr()  # produces a 5x5 table of correlation coefficients

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Left: correlation heatmap ─────────────────────────────────────────────
# Each cell shows a value between -1 and +1.
# +1 = perfect positive relationship, -1 = perfect inverse, 0 = no relationship.
sns.heatmap(
    corr,
    annot=True,       # print the number inside each cell
    fmt='.3f',        # 3 decimal places
    cmap='coolwarm',  # blue=negative correlation, red=positive
    center=0,         # white = zero correlation
    ax=axes[0],
    linewidths=0.5    # thin lines between cells for readability
)
axes[0].set_title('Correlation with log-price (numeric features only)')

# ── Right: log-price box plot by city ────────────────────────────────────
# Sort cities by their median log-price so the most expensive is on the left.
city_order = (
    df_corr.groupby('city')['log_price']
    .median()
    .sort_values(ascending=False)
    .index
)
# A box plot shows median, spread, and outliers for each city's price distribution.
sns.boxplot(
    data=df_corr, x='city', y='log_price',
    order=city_order, ax=axes[1], palette='Set2'
)
axes[1].set_title('log-Price Distribution by City')
axes[1].set_xlabel('City')

plt.tight_layout()
plt.show()

## EDA Key Findings

1. **Log-transform is mandatory** — raw price skewness > 5; log1p reduces to near-Gaussian for both sale and rental prices.
2. **All 168,446 rows are in the raw data** — For Sale (120,655 · 71.6%) and For Rent (47,791 · 28.4%). Both purposes are kept; `purpose` becomes an input feature.
3. **Sale and rental prices are on different scales** — handled by adding `purpose` as a feature; the model learns a different price level for each.
4. **5 cities only** (low cardinality): Karachi > Lahore > Islamabad > Rawalpindi > Faisalabad.
5. **7 property types in raw data; 6 used in model** — Farm House is dropped in cleaning due to high MAPE (sparse data, large area variance, unreliable pricing patterns).
6. **2 area units only**: Marla (81.8%) and Kanal (18.2%). Conversions: 1 Marla = 272.25 sqft, 1 Kanal = 5,445 sqft.
7. **Zero bedrooms/baths are missing data** — not real zeros. Will be replaced with NaN in cleaning.
8. **Lat/lon are powerful location features** — geographic clusters visible in the heatmap. Replace high-cardinality location text.
9. **Train/test split**: train = before 2019-07-01 (~60%), test = July 2019 (~40%). Time-based — no data leakage.

---
# Part 2: Data Cleaning Pipeline
---

In [ ]:
# Start with a fresh copy of the raw data — never modify df_raw directly.
# This means we can re-run the cleaning cells without re-loading the CSV.
df = df_raw.copy()
print(f'Starting rows: {len(df):,}')
print(f'Purpose split:\n{df["purpose"].value_counts()}')

# Define which columns to remove. These are identifiers and metadata
# that carry no information useful for predicting price.
#
# property_id  — just a database ID number
# location_id  — another internal ID
# page_url     — the Zameen.com listing URL
# agency       — real estate agency name (26% empty, not useful for price prediction)
# agent        — individual agent name (26% empty, not useful)
#
# NOTE: 'location' (neighbourhood name e.g. 'DHA Phase 5') is intentionally
# NOT dropped here. The model uses lat/lon internally, but the website needs
# location names for users. We'll build a lookup table from it below.
COLS_TO_DROP = ['property_id', 'location_id', 'page_url', 'agency', 'agent']
df.drop(columns=COLS_TO_DROP, inplace=True)

print(f'\nRemaining columns ({len(df.columns)}): {list(df.columns)}')

In [ ]:
# ── Text normalisation ───────────────────────────────────────────────────
# The 'location' column contains neighbourhood names scraped from Zameen.com.
# These have inconsistent casing, extra spaces, and leading/trailing whitespace
# that cause the same neighbourhood to appear as multiple different strings:
#   'DHA Phase 5 ' vs 'dha phase 5' vs 'DHA  Phase  5'
# We normalise all of these to a single canonical form so that groupby
# operations (location_lookup, target encoding) merge them correctly.

df['location'] = (
    df['location']
    .str.lower()                          # 'DHA Phase 5' -> 'dha phase 5'
    .str.strip()                          # remove leading/trailing spaces
    .str.replace(r'\s+', ' ', regex=True) # collapse multiple spaces to one
)

# city, province_name, property_type, and purpose also come from a controlled
# vocabulary but may have inconsistent casing from different scrapers.
# We apply the same normalisation to be safe.
for col in ['city', 'province_name', 'property_type', 'purpose']:
    df[col] = df[col].str.strip().str.title()  # 'for sale' -> 'For Sale'

print(f'Unique locations after normalisation: {df["location"].nunique():,}')
print(f'Sample locations: {sorted(df["location"].unique())[:5]}')

# ── Drop Farm House ───────────────────────────────────────────────────────
# Farm House listings have very high MAPE relative to all other property types.
# The category is sparse (few listings, large area variance, irregular pricing)
# and the model cannot learn reliable patterns for it. Dropping it improves
# overall model quality and avoids showing unreliable estimates to users.
n = len(df)
df = df[df['property_type'] != 'Farm House'].copy()
print(f'\nDropped {n - len(df):,} Farm House listings -> {len(df):,} remaining')

In [ ]:
def parse_area_to_sqft(area_str):
    # Conversion rates: standard Pakistani property measurements
    MARLA_TO_SQFT = 272.25
    KANAL_TO_SQFT = 5445.0  # 1 Kanal = 20 Marla = 20 x 272.25

    # If the value isn't a string (e.g. it's NaN), return NaN.
    if not isinstance(area_str, str):
        return float('nan')

    # Remove leading/trailing spaces and thousands-separator commas.
    # e.g. '4,450 Kanal' becomes '4450 Kanal'
    area_str = area_str.strip().replace(',', '')

    # Use a regular expression to extract the number and the unit.
    # Pattern breakdown:
    #   ^           = start of string
    #   (\d+\.?\d*) = one or more digits, optionally followed by a decimal
    #   \s+         = one or more spaces
    #   (Marla|Kanal) = the unit word
    #   $           = end of string
    match = re.match(r'^(\d+\.?\d*)\s+(Marla|Kanal)$', area_str, re.IGNORECASE)

    # If the string doesn't match the expected pattern, return NaN.
    if not match:
        return float('nan')

    numeric = float(match.group(1))  # the number (e.g. 5.0 from '5 Marla')
    unit    = match.group(2).lower() # the unit in lowercase ('marla' or 'kanal')

    # A zero or negative area makes no sense — treat as missing.
    if numeric <= 0:
        return float('nan')

    # Multiply by the appropriate conversion rate and return square feet.
    return numeric * (MARLA_TO_SQFT if unit == 'marla' else KANAL_TO_SQFT)

# Apply the function to every row. This creates a new numeric column.
df['area_sqft'] = df['area'].apply(parse_area_to_sqft)

failed = df['area_sqft'].isna().sum()
print(f'Area parse: {len(df) - failed:,} succeeded, {failed} failed')

# Remove rows where the area couldn't be parsed or is zero.
n = len(df)
df = df[df['area_sqft'].notna() & (df['area_sqft'] > 0)].copy()
print(f'Dropped {n - len(df)} unparseable/zero-area rows -> {len(df):,} remaining')

# Drop the original text area column — we now have area_sqft instead.
df.drop(columns=['area'], inplace=True)

In [ ]:
# ── Per-purpose IQR outlier removal ──────────────────────────────────────
# PROBLEM: sale prices (millions PKR) and rental prices (thousands PKR/month)
# create a bimodal distribution. One combined IQR fence is too wide and misses
# within-purpose outliers. SOLUTION: run IQR separately per purpose.

def remove_outliers_iqr(subset, label):
    """Apply IQR outlier removal on log-price within a single purpose group."""
    subset = subset[subset['price'].notna() & (subset['price'] > 0)].copy()
    subset['log_price'] = np.log1p(subset['price'])
    Q1, Q3 = subset['log_price'].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n = len(subset)
    subset = subset[(subset['log_price'] >= lo) & (subset['log_price'] <= hi)].copy()
    subset.drop(columns=['log_price'], inplace=True)
    print(f'{label}: kept {len(subset):,}  (dropped {n - len(subset)} outliers)')
    print(f'  Price fences: {format_pkr(np.expm1(lo))} PKR  to  {format_pkr(np.expm1(hi))} PKR')
    return subset

sale_clean = remove_outliers_iqr(df[df['purpose'] == 'For Sale'], 'For Sale')
rent_clean = remove_outliers_iqr(df[df['purpose'] == 'For Rent'], 'For Rent')
df = pd.concat([sale_clean, rent_clean], ignore_index=True)
print(f'\nTotal after per-purpose outlier removal: {len(df):,}')


In [ ]:
# ── Fix bad coordinates ──────────────────────────────────────────────────
# Pakistan's geographic bounding box: latitude 20°–40°N, longitude 55°–80°E.
# 5 rows have coordinates clearly outside Pakistan (likely swapped or zero values).
n = len(df)
df = df[df['latitude'].between(20, 40) & df['longitude'].between(55, 80)].copy()
print(f'Dropped {n - len(df)} bad-coordinate rows -> {len(df):,} remaining')

# ── Fix bedrooms and baths ────────────────────────────────────────────────
# 0 means the seller didn't fill in the field — it does NOT mean zero rooms.
# We replace 0 with NaN (Not a Number = 'missing') so the model knows
# the data is absent, not that the answer is zero.
df['bedrooms'] = df['bedrooms'].replace(0, float('nan'))

# Also cap at 20 — listings claiming 50+ bedrooms are likely data errors.
# .clip(upper=20) changes any value above 20 to exactly 20.
df['bedrooms'] = df['bedrooms'].clip(upper=20)
df['baths']    = df['baths'].replace(0, float('nan')).clip(upper=20)

print(f'\nBedroom NaN: {df["bedrooms"].isna().sum():,}  ({df["bedrooms"].isna().mean()*100:.1f}%)')
print(f'Bath NaN:    {df["baths"].isna().sum():,}  ({df["baths"].isna().mean()*100:.1f}%)')
print('(These NaN rows are kept — LightGBM handles missing values natively.)')

# ── Parse listing dates and extract time features ─────────────────────────
# Convert the date string '10/20/2018' into a proper datetime object.
# We keep date_added only for the train/test split (before/after 2019-07-01).
# year_added and month_added are NOT extracted as features — see reason below:
# The training data only covers 2018-2019. The model cannot extrapolate
# to 2026 or any future year (tree models only interpolate, never extrapolate).
# Supplying 'year=2026' would be silently treated as the nearest known year,
# adding noise not signal. Date features are dropped to avoid this.
df['date_added'] = pd.to_datetime(df['date_added'], format='%m/%d/%Y')


print(f'\nDate range: {df["date_added"].min().date()} -> {df["date_added"].max().date()}')

In [ ]:
# ── Price per square foot — EDA analysis ─────────────────────────────────
# Price per sqft is the standard normalised metric in real estate:
# it lets you compare a 5-Marla and a 10-Marla property on equal terms.
#
# NOTE: We cannot use price/sqft as a direct model feature (it contains the
# target variable 'price'). Instead, this cell is pure exploratory analysis.
# A target-encoded version (median price/sqft per location, training data only)
# is added as a model feature in Part 3.

df_eda = df_raw[df_raw['price'] > 0].copy()

def parse_area_eda(s):
    import re
    m = re.match(r'^([0-9.]+)\s+(Marla|Kanal)$', str(s).strip(), re.IGNORECASE)
    if not m: return None
    v, u = float(m.group(1)), m.group(2).lower()
    return v * (272.25 if u == 'marla' else 5445.0)

df_eda['area_sqft'] = df_eda['area'].apply(parse_area_eda)
df_eda = df_eda[df_eda['area_sqft'].notna() & (df_eda['area_sqft'] > 0)].copy()
df_eda['price_per_sqft'] = df_eda['price'] / df_eda['area_sqft']

# Keep For Sale only and clip at P95 for readability
df_sale_eda = df_eda[df_eda['purpose'] == 'For Sale'].copy()
cap = df_sale_eda['price_per_sqft'].quantile(0.95)
df_sale_eda['price_per_sqft_clipped'] = df_sale_eda['price_per_sqft'].clip(upper=cap)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: median price/sqft by city
city_ppq = (
    df_sale_eda.groupby('city')['price_per_sqft']
    .median()
    .sort_values(ascending=False)
)
city_ppq.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Median Price per Sqft by City (For Sale)')
axes[0].set_ylabel('PKR per sqft')
axes[0].tick_params(axis='x', rotation=0)
for i_bar, v in enumerate(city_ppq):
    axes[0].text(i_bar, v + 50, f'{v:,.0f}', ha='center', fontsize=8)

# Right: box plot by property type
type_order = (
    df_sale_eda.groupby('property_type')['price_per_sqft'].median()
    .sort_values(ascending=False).index
)
df_sale_eda.boxplot(
    column='price_per_sqft_clipped', by='property_type',
    ax=axes[1], vert=True, patch_artist=True
)
axes[1].set_title('Price per Sqft by Property Type (capped P95)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('')

plt.suptitle('Price per Square Foot — For Sale Listings', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Median price per sqft by city (For Sale):')
for city, ppq in city_ppq.items():
    print(f'  {city:<15}: PKR {ppq:,.0f} per sqft')


In [ ]:
# ── Sqft-per-bedroom outlier removal ────────────────────────────────────
# A property's area divided by its bedroom count reveals impossible listings.
# Verified threshold: 250 sqft per bedroom.
#
# Research basis (Pakistani real estate):
#   - 2BHK minimum in Pakistan: ~850 sqft = 425 sqft/bedroom
#   - Budget 1BHK minimum: ~550 sqft
#   - Our threshold of 250 sqft/bedroom is 41% BELOW Pakistan's market minimum
#     -> catches clear data errors (wrong units, wrong bedroom count)
#     -> preserves legitimate small properties
#
# This filter only applies where bedrooms is known (not NaN).
# Listings without a bedroom count are untouched.

SQFT_PER_BED_MIN = 250

has_bedrooms = df['bedrooms'].notna()
sqft_per_bed = df.loc[has_bedrooms, 'area_sqft'] / df.loc[has_bedrooms, 'bedrooms']
suspicious   = has_bedrooms & (sqft_per_bed < SQFT_PER_BED_MIN)

print(f'Sqft/bedroom check (threshold: {SQFT_PER_BED_MIN} sqft/bedroom):')
print(f'  Listings with known bedrooms: {has_bedrooms.sum():,}')
print(f'  Flagged as suspicious:        {suspicious.sum():,}  '
      f'({suspicious.sum()/has_bedrooms.sum()*100:.2f}% of listings with bedrooms)')

# Show a few examples before removing
if suspicious.sum() > 0:
    examples = df.loc[suspicious, ['area_sqft', 'bedrooms', 'price', 'city', 'property_type']].head(5)
    examples['sqft_per_bed'] = examples['area_sqft'] / examples['bedrooms']
    print('\nSample suspicious listings:')
    print(examples.to_string())

n = len(df)
df = df[~suspicious].copy()
print(f'\nDropped {n - len(df)} suspicious listings -> {len(df):,} remaining')


In [ ]:
# ── Premium location flag ────────────────────────────────────────────────
# In Pakistan, properties in planned housing societies (DHA, Bahria Town,
# Gulberg, Clifton) command massive premiums that lat/lon alone cannot fully
# capture — two properties 300m apart can differ 3x in price depending on
# which side of the DHA boundary they sit on.
# We extract this signal from the location name using keyword matching.
df['is_premium_location'] = df['location'].str.lower().apply(
    lambda loc: int(any(kw in str(loc) for kw in PREMIUM_KEYWORDS))
)
print(f'Premium locations: {df["is_premium_location"].sum():,}  '
      f'({df["is_premium_location"].mean()*100:.1f}% of listings)')

# ── Distance to city commercial centre (Haversine km) ────────────────────
# Properties closer to the city's main business district command higher prices.
# This captures the spatial price gradient as a continuous numeric variable.
df['dist_to_center'] = df.apply(
    lambda r: haversine_km(
        r['latitude'],  r['longitude'],
        CITY_CENTERS[r['city']][0], CITY_CENTERS[r['city']][1]
    ), axis=1
)
print(f'dist_to_center (km): mean={df["dist_to_center"].mean():.1f}  '
      f'max={df["dist_to_center"].max():.1f}')


In [ ]:
# Build a lookup table that maps each location name to its average coordinates.
#
# Why: The model uses latitude/longitude as numeric features (clean, no spelling issues).
# But users on the website will type or select a location name like 'DHA Phase 5'.
# The website looks up that name here, gets the average lat/lon, and passes
# those numbers to the model. The user never sees coordinates.
location_lookup = (
    df.groupby('location')[['latitude', 'longitude']]
    .agg(
        mean_lat=('latitude', 'mean'),       # average latitude for this location
        mean_lon=('longitude', 'mean'),       # average longitude for this location
        listing_count=('latitude', 'count')  # how many listings exist for this location
    )
    .reset_index()               # turn the groupby index back into a regular column
    .sort_values('listing_count', ascending=False)  # most common locations first
)

# Save to CSV — the website backend will load this file.
location_lookup.to_csv('location_lookup.csv', index=False)
print(f'Location lookup saved: {len(location_lookup):,} unique locations')
print(f'\nTop 10 most common locations:')
print(location_lookup.head(10).to_string(index=False))

In [ ]:
# Print a summary of what the cleaning pipeline did.
print('=' * 55)
print('CLEANING PIPELINE SUMMARY')
print('=' * 55)
print(f'Started with:  168,446 rows')
print(f'Ended with:    {len(df):,} rows  ({len(df)/168446*100:.1f}% retained)')
print(f'Columns:       {len(df.columns)} -> {list(df.columns)}')

# Show which columns still have missing values.
# After cleaning, only bedrooms and baths should have NaN (intentionally).
print(f'\nNull counts in final dataset:')
null_summary = df.isnull().sum()
print(null_summary[null_summary > 0])
print('\n(Only bedrooms and baths have NaN — intentional.)')

# Save the cleaned dataset as a CSV checkpoint.
# This means if the kernel crashes or we close the notebook,
# we can reload data_cleaned.csv without re-running all the cleaning steps.
df.to_csv('data_cleaned.csv', index=False)
print("\nCheckpoint saved to 'data_cleaned.csv'")

---
# Part 3: Model — LightGBM
We train **four** LightGBM models:
- **RMSE model** — standard regression, used for RMSE and MAPE metrics
- **P10 model** — 10th percentile (low estimate)
- **P50 model** — 50th percentile (best/median estimate)
- **P90 model** — 90th percentile (high estimate)

**Output to a user:** *'We estimate between X and Y PKR, most likely around Z PKR.'*

**Location strategy (3 layers):**
1. `latitude` / `longitude` — exact GPS position as continuous features
2. `lat_lon_cluster` — KMeans zone ID (50 clusters fit on training data), letting the model learn "zone 7 = expensive area" with a single split
3. `location_median_price` / `location_price_per_sqft` — target-encoded neighbourhood signals computed from training data only

**The website** uses `location_lookup.csv` to convert a user's typed location name into coordinates, which are then passed to the model.
---

In [ ]:
# ── Target variable ──────────────────────────────────────────────────────
df['log_price'] = np.log1p(df['price'])  # predict in log-space; convert back with expm1

# ── Log-transform area ────────────────────────────────────────────────────
# Property area is right-skewed: most listings are 5-15 Marla but some Kanals
# run to 500+ sqft, producing a long tail. log1p brings the distribution closer
# to normal so the model treats all size ranges fairly.
df['log_area_sqft'] = np.log1p(df['area_sqft'])

# ── Encode categorical columns as integers ────────────────────────────────
# province_name is EXCLUDED: it is 100% determined by city in our 5-city dataset.
# (Karachi=Sindh, Lahore/Rawalpindi/Faisalabad=Punjab, Islamabad=ICT)
# Including province alongside city adds zero information and only noise.
CAT_COLS = ['city', 'property_type', 'purpose']
for col in CAT_COLS:
    df[col] = df[col].astype('category')
    df[col + '_code'] = df[col].cat.codes
    print(f'{col}: {df[col].nunique()} categories -> {list(df[col].cat.categories)}')

# ── Define model input features ───────────────────────────────────────────
# Excluded: province_name_code (redundant), year/month_added (no extrapolation to 2026).
# location_median_price, location_price_per_sqft, lat_lon_cluster are computed
# AFTER the train/test split to prevent data leakage.
FEATURE_COLS = [
    'city_code',                 # which of the 5 cities
    'property_type_code',        # house, flat, upper portion, etc. (6 types — Farm House excluded)
    'purpose_code',              # for sale or for rent
    'baths',                     # bathrooms (NaN = unfilled; LightGBM handles natively)
    'bedrooms',                  # bedrooms (NaN = unfilled)
    'log_area_sqft',             # log(sqft) — reduces right-skew in area distribution
    'latitude',                  # GPS position
    'longitude',                 # GPS position
    'is_premium_location',       # 1 = DHA/Bahria/Gulberg/Clifton etc.
    'dist_to_center',            # km straight-line to city commercial centre
    'location_median_price',     # target-encoded: median log-price for this neighbourhood
    'location_price_per_sqft',   # target-encoded: median PKR/sqft for this neighbourhood
    'lat_lon_cluster',           # KMeans geographic zone (fit on training data only)
]
# Categorical features — LightGBM uses these for native categorical splitting
CAT_FEATURES = ['city_code', 'property_type_code', 'purpose_code', 'lat_lon_cluster']
TARGET = 'log_price'

print(f'\n{len(FEATURE_COLS)} features defined.')
print('X_train/X_test will be created after target encoding + KMeans (next cell after split).')

In [ ]:
# Split by date, not randomly — simulates real deployment (train on past, predict future).
# X_train/X_test are NOT created here; they are created in the next cell AFTER
# location target encoding is applied (to prevent leakage).
SPLIT_DATE = pd.Timestamp('2019-07-01')

df_train = df[df['date_added'] < SPLIT_DATE].copy()
df_test  = df[df['date_added'] >= SPLIT_DATE].copy()

print(f'Train: {len(df_train):,} rows  '
      f'({df_train["date_added"].min().date()} -> {df_train["date_added"].max().date()})')
print(f'Test:  {len(df_test):,} rows  '
      f'({df_test["date_added"].min().date()} -> {df_test["date_added"].max().date()})')
print(f'\nPurpose split in test set:\n{df_test["purpose"].value_counts().to_string()}')


In [ ]:
# ── Location Target Encoding ─────────────────────────────────────────────
# Replace each location name with the median log-price of all TRAINING
# listings in that neighbourhood.
#
# LEAKAGE PREVENTION (critical):
#   - Median computed ONLY on training data (df_train)
#   - Same median is then MAPPED onto test data (no test prices used)
#   - Locations in test but not in train fall back to city training median,
#     then global training median as final fallback

loc_median    = df_train.groupby('location')['log_price'].median()
city_median   = df_train.groupby('city')['log_price'].median()   # city-level fallback
global_median = df_train['log_price'].median()                    # last-resort fallback

def map_with_fallback(df_subset, loc_map, city_map, global_val):
    """Map location → median, falling back to city median, then global median."""
    mapped_loc  = df_subset['location'].map(loc_map)
    mapped_city = df_subset['city'].map(city_map)
    # np.where avoids pandas .loc[] index alignment issues across pandas versions
    result = np.where(mapped_loc.isna(), mapped_city, mapped_loc)
    result = pd.Series(result, index=df_subset.index)
    return result.fillna(global_val)

df_train['location_median_price'] = map_with_fallback(df_train, loc_median, city_median, global_median)
df_test['location_median_price']  = map_with_fallback(df_test,  loc_median, city_median, global_median)

unseen = df_test['location'].map(loc_median).isna().sum()
print(f'Training locations encoded: {loc_median.shape[0]:,}')
print(f'Test rows using city/global fallback: {unseen:,} ({unseen/len(df_test)*100:.1f}%)')
print(f'Global median price: {format_pkr(np.expm1(global_median))} PKR')

# Target-encoded price per sqft: median(price/area_sqft) per location.
df_train['raw_price_per_sqft'] = df_train['price'] / df_train['area_sqft']
loc_ppq    = df_train.groupby('location')['raw_price_per_sqft'].median()
city_ppq   = df_train.groupby('city')['raw_price_per_sqft'].median()
global_ppq = df_train['raw_price_per_sqft'].median()

def map_ppq_with_fallback(df_subset, loc_map, city_map, global_val):
    mapped_loc  = df_subset['location'].map(loc_map)
    mapped_city = df_subset['city'].map(city_map)
    result = np.where(mapped_loc.isna(), mapped_city, mapped_loc)
    result = pd.Series(result, index=df_subset.index)
    return result.fillna(global_val)

df_train['location_price_per_sqft'] = map_ppq_with_fallback(df_train, loc_ppq, city_ppq, global_ppq)
df_test['location_price_per_sqft']  = map_ppq_with_fallback(df_test,  loc_ppq, city_ppq, global_ppq)
df_train.drop(columns=['raw_price_per_sqft'], inplace=True)
print(f'Global median price/sqft: PKR {global_ppq:,.0f}')

# ── KMeans Geographic Clustering ─────────────────────────────────────────
# Raw lat/lon lets the model split properties by position, but tree models
# make axis-aligned cuts (horizontal/vertical lines), which approximates a
# curved price boundary with a staircase. A KMeans cluster ID pre-computes
# spatial zones — one split captures "everything in zone 7 = expensive area"
# instead of needing many lat AND lon splits to approximate the same boundary.
#
# Fit ONLY on training coordinates — test points are assigned to the nearest
# training centroid (standard predict, no leakage).
# 50 clusters across 5 cities ≈ 10 zones per city on average.

N_CLUSTERS = 50
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(df_train[['latitude', 'longitude']])

df_train['lat_lon_cluster'] = kmeans.predict(df_train[['latitude', 'longitude']])
df_test['lat_lon_cluster']  = kmeans.predict(df_test[['latitude', 'longitude']])

cluster_sizes = pd.Series(df_train['lat_lon_cluster']).value_counts()
print(f'\nKMeans: {N_CLUSTERS} geographic clusters')
print(f'Cluster sizes — min: {cluster_sizes.min():,}  median: {int(cluster_sizes.median()):,}  max: {cluster_sizes.max():,}')

# ── Build final X/y matrices ─────────────────────────────────────────────
# All 13 features are now ready.
X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET]
X_test  = df_test[FEATURE_COLS]
y_test  = df_test[TARGET]
print(f'\nX_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Features: {FEATURE_COLS}')

In [ ]:
# Split 10% of the training data into a validation set.
# This is used for early stopping — the model checks its performance on this
# held-out slice every few rounds and stops when it stops improving.
# This prevents overfitting (memorising training data instead of learning patterns).
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,           # 10% goes to validation
    random_state=RANDOM_STATE  # fixed seed for reproducibility
)

# ── Shared model settings ─────────────────────────────────────────────────
BASE_PARAMS = dict(
    n_estimators=500,      # maximum number of trees to build
    learning_rate=0.05,    # how much each tree corrects the previous ones (smaller = more careful)
    num_leaves=63,         # maximum number of leaf nodes per tree (controls complexity)
    min_child_samples=20,  # a leaf needs at least 20 data points (prevents overfitting)
    subsample=0.8,         # use 80% of rows randomly per tree (adds diversity)
    colsample_bytree=0.8,  # use 80% of features randomly per tree (adds diversity)
    random_state=RANDOM_STATE,
    n_jobs=-1,             # use all available CPU cores for speed
    verbose=-1             # suppress LightGBM's own training output
)

# ── Helper function to train one model ───────────────────────────────────
def train_lgbm(objective, alpha=None):
    params = BASE_PARAMS.copy()  # start with the shared settings

    if objective == 'quantile':
        # Quantile regression: instead of minimising average error,
        # minimise error at a specific percentile.
        # alpha=0.10 -> predicts the 10th percentile (low estimate)
        # alpha=0.50 -> predicts the median (best estimate)
        # alpha=0.90 -> predicts the 90th percentile (high estimate)
        params['objective'] = 'quantile'
        params['alpha']     = alpha

    m = lgb.LGBMRegressor(**params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],   # monitor performance on validation set
        categorical_feature=CAT_FEATURES,  # tell LightGBM which features are categories
        callbacks=[
            lgb.early_stopping(30, verbose=False),  # stop if no improvement for 30 rounds
            lgb.log_evaluation(0)                   # silence per-round output
        ]
    )
    return m

# ── Train all four models ─────────────────────────────────────────────────
print('Training RMSE model (standard regression for error metrics)...')
model_rmse = train_lgbm('regression')

print('Training P10 model (low end of price range)...')
model_p10 = train_lgbm('quantile', alpha=0.10)

print('Training P50 model (best single estimate)...')
model_p50 = train_lgbm('quantile', alpha=0.50)

print('Training P90 model (high end of price range)...')
model_p90 = train_lgbm('quantile', alpha=0.90)

print('\nAll four models trained.')

In [ ]:
# ── RMSE model predictions ───────────────────────────────────────────────
y_pred_log   = model_rmse.predict(X_test)
rmse_log     = np.sqrt(mean_squared_error(y_test, y_pred_log))
y_pred_price = np.expm1(y_pred_log)   # convert log-predictions back to PKR
y_test_price = np.expm1(y_test)        # convert log-actuals back to PKR
mape = np.mean(np.abs((y_test_price - y_pred_price) / y_test_price)) * 100

# ── Quantile model predictions ────────────────────────────────────────────
pred_p10 = np.expm1(model_p10.predict(X_test))  # low estimate
pred_p50 = np.expm1(model_p50.predict(X_test))  # best estimate
pred_p90 = np.expm1(model_p90.predict(X_test))  # high estimate

# What % of actual prices fall inside our [P10, P90] range? Target >= 80%.
inside = ((y_test_price >= pred_p10) & (y_test_price <= pred_p90)).mean() * 100

print('=' * 55)
print('BASELINE PAR SCORES — beat these in future iterations')
print('=' * 55)
print(f'RMSE (log-price):          {rmse_log:.4f}')
print(f'MAPE (actual PKR):         {mape:.2f}%')
print(f'P10-P90 interval coverage: {inside:.1f}%  (target >= 80%)')

# ── Confidence indicator ─────────────────────────────────────────────────
# Interval width relative to the best estimate tells us how certain the model is.
# Small interval = confident. Large interval = uncertain (rare property type,
# unusual features, or neighbourhood with few training examples).
interval_ratio = (pred_p90 - pred_p10) / np.maximum(pred_p50, 1)
high_conf = (interval_ratio < 0.5).mean() * 100
mod_conf  = ((interval_ratio >= 0.5) & (interval_ratio < 1.0)).mean() * 100
low_conf  = (interval_ratio >= 1.0).mean() * 100
print(f'\nConfidence distribution (test set):')
print(f'  High confidence  (interval < 50% of estimate):  {high_conf:.1f}%')
print(f'  Moderate         (50%-100%):                     {mod_conf:.1f}%')
print(f'  Low / uncertain  (interval > 100% of estimate): {low_conf:.1f}%')

# ── Example output as a Pakistani user would see it ──────────────────────
ex_p10 = np.percentile(pred_p10, 50)
ex_p50 = np.percentile(pred_p50, 50)
ex_p90 = np.percentile(pred_p90, 50)
ratio  = (ex_p90 - ex_p10) / max(ex_p50, 1)
conf   = ('High Confidence'     if ratio < 0.5
          else 'Moderate Confidence' if ratio < 1.0
          else 'Low Confidence')
print(f'\nExample output shown to a Pakistani user:')
print(f'  Low estimate:   {format_pkr(ex_p10)} PKR')
print(f'  Best estimate:  {format_pkr(ex_p50)} PKR')
print(f'  High estimate:  {format_pkr(ex_p90)} PKR')
print(f'  Confidence:     {conf}')
print(f'  Note: Estimates are calibrated to 2018-2019 market prices.')


In [ ]:
# ─── MODEL COMPARISON: Train 5 models ────────────────────────────────────
#
# We compare 5 models spanning three families:
#   Family 1 — Linear:         Linear Regression
#   Family 2 — Bagging:        Random Forest (many trees, independent, averaged)
#   Family 3 — Boosting:       XGBoost, LightGBM, CatBoost
#              (trees built sequentially; each one corrects the previous one's errors)
#
# Why not SVM (Support Vector Machine)?
#   SVR training complexity is O(n^2) to O(n^3). On 165,000 rows that means
#   hours of training time for a single run, and it consistently underperforms
#   boosting models on large tabular datasets. Not practical here.

import time
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# ── Impute for models that cannot handle NaN natively ────────────────────────
# Linear Regression and Random Forest require complete data — no missing values.
# XGBoost, LightGBM, and CatBoost can all handle NaN internally.
# Strategy: fill NaN with the column median (robust to outliers).
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)  # learn medians from train, apply to train
X_test_imp  = imputer.transform(X_test)        # apply same medians to test (no refitting)

# Shared result store — we'll fill this in as each model trains
model_preds = {}
model_times = {}

# ── 1. Linear Regression ─────────────────────────────────────────────────────
# Assumes a straight-line relationship between each feature and log-price.
# Used as the performance floor — any real model must beat this.
t0 = time.time()
lr = LinearRegression()
lr.fit(X_train_imp, y_train)
model_times['Linear Regression'] = time.time() - t0
model_preds['Linear Regression'] = lr.predict(X_test_imp)
print(f"[1/5] Linear Regression  done  ({model_times['Linear Regression']:.1f}s)")

# ── 2. Random Forest ─────────────────────────────────────────────────────────
# Builds 100 trees independently using random subsets of data and features,
# then averages their predictions. Good but not as fast or accurate as boosting.
t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=100,           # 100 independent trees
    random_state=RANDOM_STATE,
    n_jobs=-1                   # use all CPU cores
)
rf.fit(X_train_imp, y_train)
model_times['Random Forest'] = time.time() - t0
model_preds['Random Forest'] = rf.predict(X_test_imp)
print(f"[2/5] Random Forest       done  ({model_times['Random Forest']:.1f}s)")

# ── 3. XGBoost ───────────────────────────────────────────────────────────────
# LightGBM's closest rival. Both are gradient boosting but use different
# tree-growing strategies:
#   XGBoost: grows trees level by level (depth-first)
#   LightGBM: grows the single leaf with highest gain (leaf-wise) — usually faster
# XGBoost handles NaN natively (same as LightGBM), so we use original X_train.
t0 = time.time()
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,               # suppress XGBoost output
    enable_categorical=False   # we already encoded categoricals as integers
)
xgb.fit(X_train, y_train, verbose=False)
model_times['XGBoost'] = time.time() - t0
model_preds['XGBoost'] = xgb.predict(X_test)
print(f"[3/5] XGBoost             done  ({model_times['XGBoost']:.1f}s)")

# ── 4. LightGBM ──────────────────────────────────────────────────────────────
# Already trained above as model_rmse. Re-predict on test set — no need to retrain.
model_times['LightGBM'] = 'pre-trained'
model_preds['LightGBM'] = model_rmse.predict(X_test)
print(f"[4/5] LightGBM            done  (pre-trained, predictions reused)")

# ── 5. CatBoost ──────────────────────────────────────────────────────────────
# Yandex's gradient boosting library. Key differences from LightGBM:
#   - Uses ordered boosting to reduce prediction shift (a form of overfitting)
#   - Especially strong with categorical features
#   - Handles NaN natively
#   - Often requires less hyperparameter tuning than XGBoost or LightGBM
t0 = time.time()
cat = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=RANDOM_STATE,
    verbose=0                  # suppress CatBoost output
)
cat.fit(X_train, y_train)
model_times['CatBoost'] = time.time() - t0
model_preds['CatBoost'] = cat.predict(X_test)
print(f"[5/5] CatBoost            done  ({model_times['CatBoost']:.1f}s)")

print('\nAll 5 models trained.')


In [ ]:
# ─── PERFORMANCE COMPARISON: All 5 models across 4 metrics ──────────────
#
# Four metrics, each measuring something different:
#   RMSE (log) — Root Mean Squared Error in log-price space.
#                Penalises large errors heavily. Primary metric.
#                Lower is better.
#   MAE  (log) — Mean Absolute Error in log-price space.
#                Treats every error equally (no heavy penalty for outliers).
#                Lower is better.
#   R^2        — Coefficient of determination. How much of the variance in
#                price does the model explain? 1.0 = perfect, 0.0 = useless.
#                Higher is better.
#   MAPE (%)   — Mean Absolute Percentage Error in actual PKR.
#                Most intuitive: 'predictions are off by X% on average'.
#                Lower is better.

def compute_metrics(y_true_log, y_pred_log_vals):
    rmse = np.sqrt(mean_squared_error(y_true_log, y_pred_log_vals))
    mae  = mean_absolute_error(y_true_log, y_pred_log_vals)
    r2   = r2_score(y_true_log, y_pred_log_vals)
    y_true_pkr = np.expm1(y_true_log)
    y_pred_pkr = np.expm1(y_pred_log_vals)
    mape = np.mean(np.abs((y_true_pkr - y_pred_pkr) / y_true_pkr)) * 100
    return {'RMSE (log)': rmse, 'MAE (log)': mae, 'R2': r2, 'MAPE (%)': mape}

# Compute metrics for every model
results = {name: compute_metrics(y_test.values, preds)
           for name, preds in model_preds.items()}

# ── Printed table ─────────────────────────────────────────────────────────────
col_w = 16
header = f"{'Metric':<12}" + ''.join(f"{name:>{col_w}}" for name in results)
print('=' * (12 + col_w * len(results)))
print(header)
print('=' * (12 + col_w * len(results)))
for metric in ['RMSE (log)', 'MAE (log)', 'R2', 'MAPE (%)']:
    display = 'R2' if metric == 'R2' else metric
    row = f"{display:<12}" + ''.join(f"{results[m][metric]:>{col_w}.4f}" for m in results)
    print(row)
print('=' * (12 + col_w * len(results)))
print('RMSE / MAE / MAPE: lower is better    R2: higher is better (max = 1.0)')

# ── Visualisation: 4 subplots, one per metric ────────────────────────────────
# Each subplot is a horizontal bar chart showing all 5 models side by side.
# Colors are consistent across all subplots so you can track each model visually.

metrics_list = ['RMSE (log)', 'MAE (log)', 'R2', 'MAPE (%)']
metric_labels = ['RMSE (log)', 'MAE (log)', 'R\u00b2', 'MAPE (%)']
model_names  = list(results.keys())

# One distinct color per model — consistent across all 4 subplots
colors = {
    'Linear Regression': '#e74c3c',  # red    — weakest baseline
    'Random Forest':     '#e67e22',  # orange — middle ground
    'XGBoost':           '#9b59b6',  # purple — boosting family
    'LightGBM':          '#27ae60',  # green  — boosting family
    'CatBoost':          '#2980b9',  # blue   — boosting family
}
bar_colors = [colors[m] for m in model_names]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, metric, label in zip(axes, metrics_list, metric_labels):
    vals = [results[m][metric] for m in model_names]
    bars = ax.barh(model_names, vals, color=bar_colors, edgecolor='white', height=0.5)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Higher is better' if metric == 'R2' else 'Lower is better',
                  fontsize=9, color='grey')

    # Annotate value inside each bar
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_width() * 0.97,
            bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}',
            ha='right', va='center',
            color='white', fontsize=8, fontweight='bold'
        )

plt.suptitle(
    'Model Comparison: Linear Regression  |  Random Forest  |  XGBoost  |  LightGBM  |  CatBoost\n'
    'Evaluated on held-out test set (July 2019)',
    fontsize=12, y=1.03
)
plt.tight_layout()
plt.show()

# ── Winner summary ────────────────────────────────────────────────────────────
# Find the best model on MAPE — the most interpretable metric for end users.
best_mape_model = min(results, key=lambda m: results[m]['MAPE (%)'])
best_r2_model   = max(results, key=lambda m: results[m]['R2'])
print(f"\nBest MAPE : {best_mape_model}  ({results[best_mape_model]['MAPE (%)']:.2f}%)")
print(f"Best R2   : {best_r2_model}   (R2 = {results[best_r2_model]['R2']:.4f})")


In [ ]:
# feature_importances_ tells us how many times each feature was used
# to make a split across all 500 trees. More splits = more important feature.
importance_df = (
    pd.DataFrame({
        'feature':   FEATURE_COLS,
        'importance': model_rmse.feature_importances_
    })
    .sort_values('importance', ascending=True)  # ascending so longest bar is at top
)

fig, ax = plt.subplots(figsize=(10, 6))

# Horizontal bar chart — easier to read feature names than a vertical chart.
bars = ax.barh(
    importance_df['feature'],
    importance_df['importance'],
    color='steelblue',
    edgecolor='white'
)
ax.set_title('LightGBM Feature Importance (RMSE model, split count)', fontsize=13)
ax.set_xlabel('Importance (number of times this feature was used to split the data)')

# Add the exact importance number at the end of each bar.
for bar in bars:
    w = bar.get_width()
    ax.text(
        w + 1,                              # position: just past the end of the bar
        bar.get_y() + bar.get_height() / 2, # vertically centred on the bar
        str(int(w)),
        va='center', fontsize=9
    )
plt.tight_layout()
plt.show()

In [ ]:
# Residual = actual value minus predicted value.
# If residual > 0: we under-predicted (actual was higher than we guessed).
# If residual < 0: we over-predicted (actual was lower than we guessed).
# Ideal residuals: centred at 0, no pattern — random scatter around the zero line.
residuals = y_test.values - y_pred_log

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: Predicted vs Actual ─────────────────────────────────────────
# Each dot is one test listing. Perfect predictions would lie exactly on the red line.
# Scatter around the line = prediction error.
axes[0].scatter(y_test, y_pred_log, alpha=0.08, s=2, color='steelblue')
lims = [
    min(y_test.min(), y_pred_log.min()),
    max(y_test.max(), y_pred_log.max())
]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual log1p(price)')
axes[0].set_ylabel('Predicted log1p(price)')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

# ── Panel 2: Residuals vs Predicted ──────────────────────────────────────
# If the model is unbiased, dots should scatter randomly above and below zero.
# A funnel shape (wider at one end) = the model struggles more at certain price levels.
axes[1].scatter(y_pred_log, residuals, alpha=0.08, s=2, color='coral')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)  # the ideal zero line
axes[1].set_xlabel('Predicted log1p(price)')
axes[1].set_ylabel('Residual (actual - predicted)')
axes[1].set_title('Residuals vs Predicted')

# ── Panel 3: Residual distribution histogram ─────────────────────────────
# Should look like a bell curve centred at 0.
# A skewed histogram means the model systematically over- or under-predicts.
axes[2].hist(residuals, bins=80, color='mediumseagreen', edgecolor='white')
axes[2].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[2].set_xlabel('Residual')
axes[2].set_title(
    f'Residual Distribution\n'
    f'mean={residuals.mean():.3f}, std={residuals.std():.3f}'
)

plt.suptitle('LightGBM Baseline — Residual Analysis (Test Set)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Build a results table for the test set with the columns we want to group by.
df_results = df_test[['city', 'property_type', 'purpose']].copy()

# Compute the percentage error for each individual prediction.
# Absolute value means we don't care if we over- or under-predicted,
# just how far off we were as a percentage of the true price.
df_results['pct_error'] = np.abs((y_pred_price - y_test_price) / y_test_price) * 100

# Store predicted PKR — used by the rental yield sanity check cell below.
df_results['predicted_pkr'] = y_pred_price

# Group by city and compute the average percentage error for each.
# This tells us: which cities does the model struggle with most?
print('MAPE by city:')
print(df_results.groupby('city')['pct_error']
      .mean().sort_values().apply(lambda x: f'{x:.2f}%').to_string())

# Group by property type — does the model struggle more with flats vs houses?
print('\nMAPE by property type:')
print(df_results.groupby('property_type')['pct_error']
      .mean().sort_values().apply(lambda x: f'{x:.2f}%').to_string())

# Group by purpose — is the model more accurate for sale listings or rentals?
# This is a key check since we kept both in the same model.
print('\nMAPE by purpose (For Sale vs For Rent):')
print(df_results.groupby('purpose')['pct_error']
      .mean().sort_values().apply(lambda x: f'{x:.2f}%').to_string())

In [ ]:
# ── MAPE Cross-Tabulation Heatmap: City x Property Type ─────────────────
# A single MAPE per city hides which specific segments the model struggles with.
# e.g. the model may be accurate for Houses everywhere but terrible for
# Penthouses in Islamabad (few training examples -> high error).
# High-error cells = segments to flag as 'low confidence' on the website.

mape_matrix = (
    df_results
    .groupby(['city', 'property_type'])['pct_error']
    .mean()
    .unstack(fill_value=float('nan'))
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    mape_matrix, annot=True, fmt='.1f', cmap='RdYlGn_r',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'MAPE (%)'},
    vmin=0, vmax=50
)
ax.set_title(
    'MAPE (%) by City x Property Type\n'
    'Red = high error (model uncertain)  |  Green = low error (model confident)',
    fontsize=12
)
plt.tight_layout()
plt.show()

print('Worst-performing segments (MAPE > 40%):')
flat  = mape_matrix.stack().dropna()
worst = flat[flat > 40].sort_values(ascending=False)
if len(worst) > 0:
    for (city, ptype), val in worst.items():
        print(f'  {city} / {ptype}: {val:.1f}%')
else:
    print('  None above 40% threshold — model is broadly consistent across segments')


In [ ]:
# ── Rental Yield Sanity Check ────────────────────────────────────────────
# Pakistani residential rental yield: typically 3-5% per year.
# Formula: (median annual rent / median sale price) * 100 = yield %
#
# If our Sale and Rent predictions are consistent with this known real-world
# range, the model has learned sensible relative price levels for both purposes.
# Yields outside 2-7% indicate the model prices one purpose inconsistently,
# which would need investigation before deploying both predictions on the website.

sale_med = (
    df_results[df_results['purpose'] == 'For Sale']
    .groupby('city')['predicted_pkr'].median()
)
rent_med = (
    df_results[df_results['purpose'] == 'For Rent']
    .groupby('city')['predicted_pkr'].median()
)

# rent_med is monthly PKR. Multiply by 12 for annual, then divide by sale price.
yield_pct = (rent_med * 12 / sale_med * 100).dropna()

print('Implied annual rental yield by city (expected: 3%-5%):')
print('-' * 45)
for city, y in yield_pct.sort_values().items():
    status = 'OK' if 2.0 <= y <= 7.0 else 'CHECK -- outside expected range'
    print(f'  {city:<15}: {y:.1f}%   [{status}]')
print('-' * 45)
print('Wide deviations: model may price sale vs rent inconsistently in that city.')


In [ ]:
# ── SHAP Values: How Each Feature Affects Predictions ───────────────────
# Feature importance (earlier chart) counts how often each feature was used
# to split data. It tells us WHAT the model uses.
#
# SHAP (SHapley Additive exPlanations) tells us HOW each feature affects
# individual predictions and in which DIRECTION:
#   - Positive SHAP = feature pushed price prediction HIGHER
#   - Negative SHAP = feature pushed price prediction LOWER
#   - In the beeswarm: red dots = high feature value, blue = low feature value
#
# Expected insights for Pakistani real estate:
#   - location_median_price: strong positive effect (rich area = higher price)
#   - dist_to_center: negative effect (farther from business district = cheaper)
#   - is_premium_location: positive effect (DHA/Bahria = premium)
#   - log_area_sqft: positive (bigger = more expensive)

sample_n    = min(1000, len(X_test))  # SHAP is slow on large datasets — sample
rng         = np.random.default_rng(RANDOM_STATE)
sample_idx  = rng.choice(len(X_test), size=sample_n, replace=False)
X_sample    = X_test.iloc[sample_idx]

explainer   = shap.TreeExplainer(model_rmse)
shap_values = explainer.shap_values(X_sample)

# Bar chart: mean absolute SHAP per feature
# Units: log-price impact (more interpretable than split counts)
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False)
plt.title('Mean |SHAP| — Average log-price impact of each feature')
plt.tight_layout()
plt.show()

# Beeswarm: each dot = one prediction from the sample
# Shows direction AND distribution of each feature's effect simultaneously
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, show=False)
plt.title('SHAP Value Distribution — Direction and magnitude per feature')
plt.tight_layout()
plt.show()


---
# Part 4: Model Serialization
Save all trained artifacts to `model_artifacts.pkl` — loaded once by the FastAPI backend at startup. No notebook environment needed at serve time.

**Artifacts packed:**
- 4 LightGBM models (`model_rmse`, `model_p10`, `model_p50`, `model_p90`)
- KMeans geographic clustering object (50 clusters)
- Location target encoding maps (`loc_median`, `loc_ppq`) with city and global fallbacks
- Category → integer encodings for `city`, `property_type`, `purpose`
- Feature metadata (`FEATURE_COLS`, `CAT_FEATURES`)
- Domain constants (`CITY_CENTERS`, `PREMIUM_KEYWORDS`)
---

In [ ]:
import joblib, os

# ── Build the artifacts bundle ────────────────────────────────────────────
# Everything the inference wrapper needs to reproduce a prediction from scratch.
# Loaded once at API startup — no notebook environment needed at serve time.

# Extract category → integer mappings from pandas cat.codes order.
# Must match exactly how the notebook encoded them (sorted unique values → 0, 1, 2...).
city_enc          = {cat: i for i, cat in enumerate(df['city'].cat.categories)}
property_type_enc = {cat: i for i, cat in enumerate(df['property_type'].cat.categories)}
purpose_enc       = {cat: i for i, cat in enumerate(df['purpose'].cat.categories)}

artifacts = {
    # ── Models ──────────────────────────────────────────────────────────────
    'model_p10':  model_p10,
    'model_p50':  model_p50,
    'model_p90':  model_p90,
    'model_rmse': model_rmse,

    # ── Geographic clustering ────────────────────────────────────────────────
    'kmeans': kmeans,

    # ── Location target encoding (computed from training data only) ──────────
    'loc_median':    loc_median,    # location name → median log-price
    'city_median':   city_median,   # city → median log-price (fallback tier 2)
    'global_median': global_median, # global median (fallback tier 3)
    'loc_ppq':       loc_ppq,       # location name → median PKR/sqft
    'city_ppq':      city_ppq,      # city → median PKR/sqft (fallback tier 2)
    'global_ppq':    global_ppq,    # global median PKR/sqft (fallback tier 3)

    # ── Categorical encodings ────────────────────────────────────────────────
    'city_enc':          city_enc,
    'property_type_enc': property_type_enc,
    'purpose_enc':       purpose_enc,

    # ── Feature metadata ─────────────────────────────────────────────────────
    'FEATURE_COLS': FEATURE_COLS,
    'CAT_FEATURES': CAT_FEATURES,

    # ── Domain constants ─────────────────────────────────────────────────────
    'CITY_CENTERS':     CITY_CENTERS,
    'PREMIUM_KEYWORDS': PREMIUM_KEYWORDS,
}

OUTPUT_PATH = 'model_artifacts.pkl'
joblib.dump(artifacts, OUTPUT_PATH, compress=3)

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f'Saved -> {OUTPUT_PATH}  ({size_mb:.1f} MB)')
print(f'\nModels packed:      model_rmse, model_p10, model_p50, model_p90')
print(f'KMeans clusters:    {kmeans.n_clusters}')
print(f'Locations encoded:  {len(loc_median):,}')
print(f'Features:           {len(FEATURE_COLS)}')
print(f'\nCategory encodings:')
print(f'  city:          {city_enc}')
print(f'  property_type: {property_type_enc}')
print(f'  purpose:       {purpose_enc}')